# Lesson 3: nn.Module, nn.Linear and optimizers

Goes with the Lesson 3 page of the Forkcast course. The model becomes an `nn.Module`, the update becomes an optimizer, the features get normalized, and the training loop moves into `forkcast/train.py` for good.

Runtime: CPU is enough.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. Meet nn.Linear

In [ ]:
import torch
from torch import nn
from forkcast.data import make_orders, FEATURES, TRUE_WEIGHTS, TRUE_BIAS
X, y = make_orders(400, seed=0)

layer = nn.Linear(4, 1)
print(layer.weight.shape, layer.bias.shape)
print(layer.weight)
print(layer(X).shape)                                   # note the trailing 1
print(torch.allclose(layer(X), X @ layer.weight.T + layer.bias))

## 3. The model: `forkcast/model.py`

Two lines in `__init__`, one in `forward`. Re-run the cell after editing.

In [ ]:
%%writefile forkcast/model.py
"""Lesson 3. The delivery-time model as an nn.Module.

Replace every ``raise NotImplementedError`` with one to three lines of code.
"""

from __future__ import annotations

import torch
from torch import Tensor, nn


class LinearRegressor(nn.Module):
    """minutes = X @ w + b, with the knobs owned by an nn.Linear layer."""

    def __init__(self, n_features: int):
        super().__init__()
        raise NotImplementedError("Lesson 3, step 3: self.linear = nn.Linear(n_features, 1)")

    def forward(self, X: Tensor) -> Tensor:
        """(N, d) -> (N,). nn.Linear returns (N, 1); squeeze the last dimension away."""
        raise NotImplementedError("Lesson 3, step 3: return self.linear(X).squeeze(-1)")

In [ ]:
import importlib
from forkcast import model as model_module
importlib.reload(model_module)
from forkcast.model import LinearRegressor

m = LinearRegressor(4)
print(m(X).shape)                                        # torch.Size([400]) -- squeezed
print(torch.allclose(m(X), X @ m.linear.weight[0] + m.linear.bias[0]))
print(sum(p.numel() for p in m.parameters()), "parameters")

## 4. The normalizer: `forkcast/preprocess.py`

In [ ]:
%%writefile forkcast/preprocess.py
"""Lesson 3. Feature normalization with statistics learned from training data only."""

from __future__ import annotations

import torch
from torch import Tensor


class Normalizer:
    """Give every feature column mean 0 and standard deviation 1.

    ``fit`` learns the per-column mean and std from one tensor (the training
    data). ``transform`` applies those stored statistics to any tensor, so
    validation and test data are scaled the same way as the training data.
    """

    def __init__(self):
        self.mean: Tensor | None = None
        self.std: Tensor | None = None

    def fit(self, X: Tensor) -> "Normalizer":
        """Store the column means and population stds (``correction=0``) of X, shape (d,) each.

        Clamp the std with ``.clamp_min(1e-6)`` so a constant column cannot cause a division by zero.
        Return ``self`` so ``Normalizer().fit(X)`` works.
        """
        raise NotImplementedError("Lesson 3, step 4: self.mean = ..., self.std = ...; return self")

    def transform(self, X: Tensor) -> Tensor:
        """(N, d) -> (N, d) using the stored statistics. Must not recompute them."""
        raise NotImplementedError("Lesson 3, step 4: (X - self.mean) / self.std")

    def fit_transform(self, X: Tensor) -> Tensor:
        return self.fit(X).transform(X)

In [ ]:
from forkcast import preprocess
importlib.reload(preprocess)
from forkcast.preprocess import Normalizer

norm = Normalizer().fit(X)
print(norm.mean, norm.std)
Xn = norm.transform(X)
print(Xn.mean(dim=0), Xn.std(dim=0, correction=0))      # zeros and ones

## 5. The toolkit: `forkcast/train.py`

`train_one_epoch`, `evaluate`, `fit`, `save`, `load`. The docstrings give the order of the lines.

In [ ]:
%%writefile forkcast/train.py
"""Lesson 3. The training toolkit: one epoch, evaluation, fit, save, load.

These functions are reused by every later lesson. Right now the whole dataset
fits in one batch; Lesson 6 replaces X, y with a DataLoader.
"""

from __future__ import annotations

from pathlib import Path

import torch
from torch import Tensor, nn


def train_one_epoch(model: nn.Module, X: Tensor, y: Tensor, optimizer: torch.optim.Optimizer, loss_fn: nn.Module) -> float:
    """One pass over the data. Returns the loss as a Python float.

    The three lines every PyTorch training step has, in this order:
    zero the gradients, backward, step. Put the model in training mode first.
    """
    raise NotImplementedError("Lesson 3, step 5: model.train(); zero_grad; forward; loss; backward; step; return loss.item()")


@torch.no_grad()
def evaluate(model: nn.Module, X: Tensor, y: Tensor, loss_fn: nn.Module) -> float:
    """Loss on data the model is not learning from. No gradients are needed or wanted."""
    raise NotImplementedError("Lesson 3, step 5: model.eval(); return loss_fn(model(X), y).item()")


def fit(model: nn.Module, X: Tensor, y: Tensor, epochs: int = 300, lr: float = 0.1,
        optimizer_cls=torch.optim.SGD, loss_fn: nn.Module | None = None) -> list[float]:
    """Build the optimizer over ``model.parameters()``, run ``epochs`` epochs, return one loss per epoch."""
    raise NotImplementedError("Lesson 3, step 5: optimizer = optimizer_cls(model.parameters(), lr=lr); loop train_one_epoch")


def save(model: nn.Module, path: str | Path) -> None:
    """Write the model's state_dict to ``path``."""
    raise NotImplementedError("Lesson 3, step 8: torch.save(model.state_dict(), path)")


def load(model: nn.Module, path: str | Path) -> nn.Module:
    """Load a state_dict from ``path`` into ``model`` (built with the same n_features) and return it."""
    raise NotImplementedError("Lesson 3, step 8: model.load_state_dict(torch.load(path, weights_only=True)); return model")

## 6. Fit on normalized features, and read what it learned

In [ ]:
from forkcast import train
importlib.reload(train)

torch.manual_seed(0)
model = LinearRegressor(4)
losses = train.fit(model, Xn, y, epochs=300, lr=0.1)
print(f"loss {losses[0]:.1f} -> {losses[-1]:.2f}")
print("normalized-scale weights:", model.linear.weight[0].detach(), "bias:", model.linear.bias.detach())

w, b = model.linear.weight[0].detach(), model.linear.bias[0].detach()
print("original-scale weights:", w / norm.std, "   truth", TRUE_WEIGHTS)
print("original-scale bias:   ", (b - (w * norm.mean / norm.std).sum()).item(), "   truth", TRUE_BIAS)

import matplotlib.pyplot as plt
plt.plot(losses); plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss (log scale)");

### Break it on purpose

The same learning rate on raw features, with SGD and then with Adam.

In [ ]:
torch.manual_seed(0)
raw = train.fit(LinearRegressor(4), X, y, epochs=60, lr=0.1)
print("raw features, SGD lr=0.1:", [round(v, 1) for v in raw[:4]], "...", raw[-1])

torch.manual_seed(0)
adam = train.fit(LinearRegressor(4), X, y, epochs=300, lr=0.1, optimizer_cls=torch.optim.Adam)
print("raw features, Adam lr=0.1, 300 epochs:", round(adam[-1], 2))

## 7. Save and load

In [ ]:
train.save(model, "linear.pt")
fresh = train.load(LinearRegressor(4), "linear.pt")
print(torch.equal(fresh(Xn), model(Xn)))
print(list(model.state_dict().keys()))

## 8. Definition of done

Expected: `7 passed`, then `27 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_03.py -q

In [ ]:
!pytest -m fast -q

## 9. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 3: nn.Module, optimizers, normalization"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main